In [1]:
import csv
import sys
import pandas as pd
import random
from proposed_design_types import PROPOSED_DESIGN_TYPES

In [2]:

filename = 'pubmed_articles_neg.csv'

p = 1  # 1% of the lines
# keep the header, then take only 1% of lines
# if random from [0,1] interval is greater than 0.01 the row will be skipped
neg_data = pd.read_csv(
         filename,
         header=0, 
         skiprows=lambda i: i>0 and random.random() > p
)

In [ ]:
neg_data.sample(n=3, random_state=1)

In [30]:
neg_data.head()

,Unnamed: 0,pmid,status,indexing_method,title,abstract,abstract_plaintext,authors,journal,journal_abbrv,year,mesh,month,volume,issue,pages,ptyp,registry_ids,dois
0,23,18222623,MEDLINE,Human,A comparison of right and extended right hepat...,"[{'header': 'AIMS', 'text': 'Colorectal liver ...",AIMS\n\n\nColorectal liver metastases are trea...,"[{'Initials': 'ND', 'LastName': 'Karanjia', 'F...",European journal of surgical oncology : the jo...,Eur J Surg Oncol,2009.0,"['Adult', 'Aged', 'Aged, 80 and over', 'Colore...",Jan,35,1,"{'page_from': '65', 'page_to': '70'}","['Comparative Study', 'Journal Article']",[],['10.1016/j.ejso.2007.12.002']
1,72,18222671,MEDLINE,Human,Aescin and troxerutin as a successful combinat...,"[{'header': '_UNSTRUCTURED', 'text': 'A fixed ...",A fixed combination of aescin and troxerutin h...,"[{'Initials': 'CP', 'LastName': 'Siegers', 'Fo...",Phytomedicine : international journal of phyto...,Phytomedicine,2008.0,"['Administration, Oral', 'Aesculus', 'Aged', '...",Mar,15,3,"{'page_from': '160', 'page_to': '163'}","['Comparative Study', 'Journal Article', 'Rand...",[],['10.1016/j.phymed.2007.11.025']
2,832,18223488,MEDLINE,Human,Tonsillectomy does not decrease the risk of He...,[],NaN,"[{'Initials': 'CO', 'LastName': 'Kara', 'ForeN...",Journal of clinical gastroenterology,J Clin Gastroenterol,2008.0,"['Disease Transmission, Infectious', 'Helicoba...",Mar,42,3,"{'page_from': '326', 'page_to': '327'}","['Letter', 'Comment']",[],['10.1097/01.mcg.0000247990.04463.82']
3,1698,18224403,MEDLINE,Human,Current status of accelerated partial breast i...,"[{'header': '_UNSTRUCTURED', 'text': 'Accelera...",Accelerated partial breast irradiation (APBI) ...,"[{'Initials': 'M', 'LastName': 'Mitsumori', 'F...","Breast cancer (Tokyo, Japan)",Breast Cancer,2008.0,"['Breast Neoplasms', 'Female', 'Humans', 'Mast...",NaN,15,1,"{'page_from': '101', 'page_to': '107'}","['Journal Article', 'Review']",[],['10.1007/s12282-007-0012-1']
4,1745,18224455,MEDLINE,Human,Effect of paradoxical sleep deprivation on oxi...,"[{'header': '_UNSTRUCTURED', 'text': 'The pres...",The present study examined the effects of para...,"[{'Initials': 'R', 'LastName': 'Singh', 'ForeN...",Biogerontology,Biogerontology,2008.0,"['Aging', 'Animals', 'Brain', 'Glutathione', '...",Jun,9,3,"{'page_from': '153', 'page_to': '162'}","['Comparative Study', 'Journal Article', ""Rese...",[],['10.1007/s10522-008-9124-z']


In [3]:
len(pos_data)

92277

In [2]:
import tqdm
import json
import sys
import os
import math
import pandas as pd
import tempfile
import csv
import gzip
import xml.etree.cElementTree as ET
import glob
import random
from readers import pmreader2 as pmreader
from proposed_design_types import PROPOSED_DESIGN_TYPES


def abstract_plaintext(abstract):
    out = []
    for sec in abstract:
        if sec["header"] != "_UNSTRUCTURED":
            out.append(sec["header"])
        out.append(sec["text"])

    return "\n".join(out)

def iter_abstracts(filename, num_to_select = None):
    with open(filename, 'rb') as f:
        decompressedFile = gzip.GzipFile(fileobj=f, mode='rb')
        xml_dump = decompressedFile.read()
        xml_parsed = ET.fromstring(xml_dump)
        articles = xml_parsed.findall('PubmedArticle/MedlineCitation')
        if num_to_select:
            article_sample = random.sample(articles, num_to_select)
        else:
            article_sample = articles
        for j, article in enumerate(tqdm.tqdm(article_sample)):
            p_article = pmreader.PubmedCorpusReader(xml_ET=article)
            yield p_article.to_dict()
            
def is_proposed_design_type(ptyps):
    return set(ptyps).intersection(set(PROPOSED_DESIGN_TYPES)) != set()

            


In [4]:

    
#     num_to_select = None

#     filenames = glob.glob('/data/iain/pubmed/*.gz')
    
#     pos_data = pd.DataFrame()


#     num_todo = len(filenames)
    
    
#     for i, fn in tqdm.tqdm(enumerate(filenames)): # this line does *all* >900 files (takes ages)
#     # for id in range(len(filenames)):
#     # for i, fn in tqdm.tqdm(enumerate([filenames[900]])): # file 900 looks to me to be a decent example; perhaps others better! but some earlier abstracts don't contain much info


#             # here we're iterating through *all* files
#             # in the final version, probably we'd want to randomly sample, to generate a manageable amount of data
#             # I suspect we might not have enough RAM to hold all 300M articles at once, (and so you will need to sample in the loop)
#             # but possibly it might work without this, and you could sample from the Pandas dataframe below instead!
#             sys.__stdout__.write("file number {}/{} ({}) processing \n".format(i, num_todo, fn)) # optional to see progress from terminal
#             print("file number {}/{} ({}) processing \n".format(i, num_todo, fn))
#             data = []
            
#             for article in iter_abstracts(fn, num_to_select):
#                     data.append(article)

#             data_pd = pd.DataFrame(data)
                
#             pos_data_min = data_pd[data_pd['ptyp'].apply(is_proposed_design_type)]
#             pos_data = pd.concat([pos_data, pos_data_min])
                
#             pos_data.to_csv('pubmed_articles_pos.csv')



In [13]:
pos_data.head()

,Unnamed: 0,pmid,status,indexing_method,title,abstract,abstract_plaintext,authors,journal,journal_abbrv,year,mesh,month,volume,issue,pages,ptyp,registry_ids,dois
0,435,18223039,MEDLINE,Human,What's new in hypertension 2007?,[],NaN,"[{'Initials': 'JF', 'LastName': 'Mann', 'ForeN...","Nephrology, dialysis, transplantation : offici...",Nephrol Dial Transplant,2008,"['Animals', 'Humans', 'Hypertension']",Feb,23,2,"{'page_from': '466', 'page_to': '470'}","['Journal Article', 'Review']",[],['10.1093/ndt/gfm904']
1,985,18223674,MEDLINE,Human,Procoagulant signalling mechanisms in lung inf...,"[{'header': '_UNSTRUCTURED', 'text': 'There is...",There is compelling evidence that uncontrolled...,"[{'Initials': 'RC', 'LastName': 'Chambers', 'F...",British journal of pharmacology,Br J Pharmacol,2008,"['Animals', 'Anticoagulants', 'Blood Coagulati...",Mar,153 Suppl 1,NaN,"{'page_from': 'S367', 'page_to': 'S378'}","['Journal Article', ""Research Support, Non-U.S...",[],['10.1038/sj.bjp.0707603']
2,1585,18224283,MEDLINE,Human,Metabolic flux analysis in biotechnology proce...,"[{'header': '_UNSTRUCTURED', 'text': 'Metaboli...",Metabolic flux analysis (MFA) has become a fun...,"[{'Initials': 'S', 'LastName': 'Iwatani', 'For...",Biotechnology letters,Biotechnol Lett,2008,"['Amino Acids', 'Anti-Bacterial Agents', 'Carb...",May,30,5,"{'page_from': '791', 'page_to': '799'}","['Journal Article', 'Review']",[],['10.1007/s10529-008-9633-5']
3,2645,18225393,MEDLINE,Human,Prognostic significance of the histomorphometr...,"[{'header': 'OBJECTIVE', 'text': 'To study the...",OBJECTIVE\n\n\nTo study the correlation of his...,"[{'Initials': 'A', 'LastName': 'Lamba', 'ForeN...",Analytical and quantitative cytology and histo...,Anal Quant Cytol Histol,2007,"['Adolescent', 'Adult', 'Biopsy', 'Bone Marrow...",Dec,29,6,"{'page_from': '370', 'page_to': '376'}","['Comparative Study', 'Journal Article']",[],[]
4,2648,18225396,MEDLINE,Human,Immunohistochemical markers in the evaluation ...,[],NaN,"[{'Initials': 'MJ', 'LastName': 'Requena', 'Fo...",Analytical and quantitative cytology and histo...,Anal Quant Cytol Histol,2007,"['Biomarkers, Tumor', 'Carcinoma, Squamous Cel...",Dec,29,6,"{'page_from': '380', 'page_to': '382'}","['Comment', 'Letter']",[],[]


In [14]:
set(pos_data['indexing_method'])

{'Automated', 'Curated', 'Human'}

In [11]:
print(len(pos_data[pos_data['indexing_method'] =='Human']))
print(len(pos_data[pos_data['indexing_method'] =='Curated']))
print(len(pos_data[pos_data['indexing_method'] =='Automated']))

85562
2370
4358


In [22]:
for des_typ in PROPOSED_DESIGN_TYPES:
    print(des_typ, len(pos_data[(pos_data['indexing_method'] =='Human') & ([des_typ in typs for typs in pos_data['ptyp']])]))

Editorial 4656
Address 45
Letter 9335
Lecture 85
Adaptive Clinical Trial 0
Clinical Trial 5802
Clinical Trial, Phase I 647
Clinical Trial, Phase II 482
Clinical Trial, Phase III 172
Clinical Trial, Phase IV 20
Clinical Trial Protocol 32
Consensus Development Conference 130
Consensus Development Conference, NIH 14
Guideline 354
Practice Guideline 286
Case Reports 20594
Controlled Clinical Trial 937
Observational Study 801
Randomized Controlled Trial 4861
Comment 5105
Comparative Study 18302
Equivalence Trial 6
Evaluation Study 2508
Randomized Controlled Trial 4861
Government Publication 1
Meta-Analysis 1143
Personal Narrative 55
Pragmatic Clinical Trial 11
Review 26526
Systematic Review 1236
Validation Study 949


In [23]:
for des_typ in PROPOSED_DESIGN_TYPES:
    print(des_typ, len(pos_data[(pos_data['indexing_method'] =='Curated') & ([des_typ in typs for typs in pos_data['ptyp']])]))

Editorial 61
Address 2
Letter 51
Lecture 0
Adaptive Clinical Trial 0
Clinical Trial 88
Clinical Trial, Phase I 17
Clinical Trial, Phase II 13
Clinical Trial, Phase III 5
Clinical Trial, Phase IV 0
Clinical Trial Protocol 1
Consensus Development Conference 3
Consensus Development Conference, NIH 0
Guideline 7
Practice Guideline 7
Case Reports 410
Controlled Clinical Trial 17
Observational Study 96
Randomized Controlled Trial 270
Comment 0
Comparative Study 300
Equivalence Trial 1
Evaluation Study 120
Randomized Controlled Trial 270
Government Publication 0
Meta-Analysis 109
Personal Narrative 0
Pragmatic Clinical Trial 0
Review 1008
Systematic Review 181
Validation Study 66


In [25]:
for des_typ in PROPOSED_DESIGN_TYPES:
    print(des_typ, len(pos_data[(pos_data['indexing_method'] =='Automated') & ([des_typ in typs for typs in pos_data['ptyp']])]))

Editorial 743
Address 0
Letter 1714
Lecture 1
Adaptive Clinical Trial 0
Clinical Trial 1
Clinical Trial, Phase I 1
Clinical Trial, Phase II 1
Clinical Trial, Phase III 1
Clinical Trial, Phase IV 0
Clinical Trial Protocol 0
Consensus Development Conference 0
Consensus Development Conference, NIH 0
Guideline 0
Practice Guideline 0
Case Reports 288
Controlled Clinical Trial 0
Observational Study 4
Randomized Controlled Trial 14
Comment 3731
Comparative Study 1
Equivalence Trial 0
Evaluation Study 0
Randomized Controlled Trial 14
Government Publication 0
Meta-Analysis 15
Personal Narrative 0
Pragmatic Clinical Trial 0
Review 142
Systematic Review 15
Validation Study 0


In [26]:
# set(pos_data['indexing_method'])

{'Automated', 'Curated', 'Human'}

In [ ]:
check = read_csv()

In [2]:
field_size_limit = sys.maxsize
csv.field_size_limit(field_size_limit)

131072

In [3]:
pos_data = pd.read_csv('pubmed_articles_pos.csv', engine='python', error_bad_lines=False)

FileNotFoundError: [Errno 2] No such file or directory: 'pubmed_articles_pos.csv'

In [24]:
train_data = pd.read_csv('pubmed_articles_train.csv', engine='python', error_bad_lines=False)

In [25]:
len(train_data)

8027357

In [26]:
train_data

,Unnamed: 0,title,abstract,authors,journal,journal_abbrv,year,mesh,month,volume,issue,pages,ptyp,pmid,registry_ids,dois
0,0,Expression and prognostic relevance of annexin...,"[{'header': 'OBJECTIVES', 'text': 'By differen...","[{'Initials': 'J', 'LastName': 'Köllermann', '...",European urology,Eur Urol,2008,"['Aged', 'Annexin A3', 'Humans', 'Male', 'Midd...",Dec,54,6,"{'page_from': '1314', 'page_to': '1323'}","['Journal Article', ""Research Support, Non-U.S...",18222597,[],['10.1016/j.eururo.2008.01.001']
1,1,Laparoscopic versus open partial nephrectomy: ...,"[{'header': 'OBJECTIVES', 'text': 'To critical...","[{'Initials': 'F', 'LastName': 'Porpiglia', 'F...",European urology,Eur Urol,2008,"['Carcinoma, Renal Cell', 'Clinical Competence...",Apr,53,4,{},"['Journal Article', 'Review']",18222599,[],['10.1016/j.eururo.2008.01.025']
2,2,Celecoxib has potent antitumour effects as a s...,"[{'header': 'OBJECTIVES', 'text': 'Prostagland...","[{'Initials': 'SJ', 'LastName': 'Dovedi', 'For...",European urology,Eur Urol,2008,"['Analysis of Variance', 'Animals', 'BCG Vacci...",Sep,54,3,"{'page_from': '621', 'page_to': '6230'}","['Journal Article', ""Research Support, Non-U.S...",18222600,[],['10.1016/j.eururo.2008.01.013']
3,3,Editorial comment on: expression and prognosti...,[],"[{'Initials': 'P', 'LastName': 'Härkönen', 'Fo...",European urology,Eur Urol,2008,"['Annexin A3', 'Humans', 'Male', 'Prognosis', ...",Dec,54,6,"{'page_from': '1322', 'page_to': '1322'}","['Comment', 'Editorial']",18222601,[],['10.1016/j.eururo.2008.01.002']
4,4,Editorial comment on: celecoxib has potent ant...,[],"[{'Initials': 'S', 'LastName': 'Brandau', 'For...",European urology,Eur Urol,2008,"['Animals', 'BCG Vaccine', 'Carcinoma, Transit...",Sep,54,3,"{'page_from': '629', 'page_to': '6230'}","['Comment', 'Editorial']",18222602,[],['10.1016/j.eururo.2008.01.014']
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8027352,29995,A simple and sensitive antigen retrieval metho...,"[{'header': '_UNSTRUCTURED', 'text': 'The mask...","[{'Initials': 'Y', 'LastName': 'Jiao', 'ForeNa...",Journal of neuroscience methods,J Neurosci Methods,1999,"['Animals', 'Antigens', 'Brain', 'Citrates', '...",Nov,93,2,"{'page_from': '149', 'page_to': '1462'}","['Journal Article', ""Research Support, Non-U.S...",10634500,[],[]
8027353,29996,An eyecup preparation for the rat and mouse.,"[{'header': '_UNSTRUCTURED', 'text': 'The eyec...","[{'Initials': 'EA', 'LastName': 'Newman', 'For...",Journal of neuroscience methods,J Neurosci Methods,1999,"['Animals', 'Dark Adaptation', 'Electroretinog...",Nov,93,2,"{'page_from': '169', 'page_to': '1675'}","['Journal Article', ""Research Support, U.S. Go...",10634502,[],[]
8027354,29997,Comparison of tRNA activity under homologous a...,"[{'header': '_UNSTRUCTURED', 'text': ""Rate of ...","[{'Initials': 'M', 'LastName': 'Vinayak', 'For...",Molecular biology reports,Mol Biol Rep,1999,"['Algal Proteins', 'Amino Acyl-tRNA Synthetase...",Dec,26,4,"{'page_from': '217', 'page_to': '2122'}","['Comparative Study', 'Journal Article', ""Rese...",10634503,[],[]
8027355,29998,Differential binding of NF1 transcription fact...,"[{'header': '_UNSTRUCTURED', 'text': 'Differen...","[{'Initials': 'BK', 'LastName': 'Nayak', 'Fore...",Molecular biology reports,Mol Biol Rep,1999,"['Breast Neoplasms', 'DNA Footprinting', 'DNA-...",Dec,26,4,"{'page_from': '223', 'page_to': '2230'}","['Journal Article', ""Research Support, Non-U.S...",10634504,[],[]


In [33]:
val_data = pd.read_csv('pubmed_articles_val.csv', engine='python', error_bad_lines=False)

In [34]:
len(val_data)

615318

In [35]:
val_data

,Unnamed: 0,title,abstract,authors,journal,journal_abbrv,year,mesh,month,volume,issue,pages,ptyp,pmid,registry_ids,dois
0,0,Targeted individual prophylaxis offers superio...,"[{'header': '_UNSTRUCTURED', 'text': 'Cytomega...","[{'Initials': 'S', 'LastName': 'Sood', 'ForeNa...",Liver transplantation : official publication o...,Liver Transpl,2015,"['Adult', 'Cytomegalovirus Infections', 'Human...",Dec,21,12,"{'page_from': '1478', 'page_to': '1485'}","['Journal Article', ""Research Support, Non-U.S...",26194446,[],['10.1002/lt.24216']
1,1,Experimental model for controlling shear using...,"[{'header': 'OBJECTIVE', 'text': 'This study h...","[{'Initials': 'Y', 'LastName': 'Jabbar', 'Fore...","Clinical biomechanics (Bristol, Avon)","Clin Biomech (Bristol, Avon)",2015,"['Biomechanical Phenomena', 'Equipment Design'...",Nov,30,9,"{'page_from': '995', 'page_to': '91001'}","['Journal Article', ""Research Support, Non-U.S...",26194448,[],['10.1016/j.clinbiomech.2015.06.017']
2,2,Pre-transplant comorbidity burden and post-tra...,"[{'header': '_UNSTRUCTURED', 'text': 'The Haem...","[{'Initials': 'JE', 'LastName': 'Vaughn', 'For...",British journal of haematology,Br J Haematol,2015,"['Adolescent', 'Adult', 'Aged', 'Allografts', ...",Nov,171,3,"{'page_from': '411', 'page_to': '46'}","['Clinical Trial', 'Journal Article', 'Multice...",26194447,[],['10.1111/bjh.13591']
3,4,Intracardiac echocardiography for percutaneous...,[],"[{'Initials': 'T', 'LastName': 'Onishi', 'Fore...",Journal of echocardiography,J Echocardiogr,2015,"['Cardiomyopathy, Hypertrophic', 'Catheter Abl...",Dec,13,4,"{'page_from': '151', 'page_to': '12'}",['Journal Article'],26194450,[],['10.1007/s12574-015-0256-2']
4,5,Input-output analysis on the economic impact o...,"[{'header': 'OBJECTIVES', 'text': ""Since the C...","[{'Initials': 'G', 'LastName': 'Yamada', 'Fore...",Environmental health and preventive medicine,Environ Health Prev Med,2015,"['Health Care Costs', 'Humans', 'Japan', 'Mode...",Sep,20,5,"{'page_from': '379', 'page_to': '3787'}",['Journal Article'],26194451,[],['10.1007/s12199-015-0478-y']
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
615313,29994,Bioinspired locomotion and grasping in water: ...,"[{'header': '_UNSTRUCTURED', 'text': 'The octo...","[{'Initials': 'M', 'LastName': 'Cianchetti', '...",Bioinspiration & biomimetics,Bioinspir Biomim,2015,"['Animals', 'Computer Simulation', 'Equipment ...",May,10,3,"{'page_from': '035003', 'page_to': '035003'}","['Journal Article', ""Research Support, Non-U.S...",25970014,[],['10.1088/1748-3190/10/3/035003']
615314,29996,Is the kinetoplast DNA a percolating network o...,"[{'header': '_UNSTRUCTURED', 'text': 'In this ...","[{'Initials': 'D', 'LastName': 'Michieletto', ...",Physical biology,Phys Biol,2015,"['DNA, Kinetoplast', 'Models, Theoretical', 'N...",May,12,3,"{'page_from': '036001', 'page_to': '036001'}","['Journal Article', ""Research Support, Non-U.S...",25970016,[],['10.1088/1478-3975/12/3/036001']
615315,29997,Unmyelinated tactile cutaneous nerves signal e...,"[{'header': 'INTRODUCTION', 'text': 'Intrapers...","[{'Initials': 'EH', 'LastName': 'Jönsson', 'Fo...",The journal of sexual medicine,J Sex Med,2015,"['Adult', 'Affect', 'Arousal', 'Emotions', 'Fe...",Jun,12,6,"{'page_from': '1338', 'page_to': '1345'}","['Journal Article', ""Research Support, Non-U.S...",25970018,[],['10.1111/jsm.12905']
615316,29998,Multivalent Antigens for Promoting B and T Cel...,"[{'header': '_UNSTRUCTURED', 'text': 'Efficaci...","[{'Initials': 'NR', 'LastName': 'Bennett', 'Fo...",ACS chemical biology,ACS Chem Biol,2015,"['Amino Acid Sequence', 'Antigens', 'B-Lymphoc...",Aug,10,8,"{'page_from': '1817', 'page_to': '1824'}","['Journal Article', 'Research Support, N.I.H.,...",25970017,[],['10.1021/acschembio.5b00239']


In [36]:
test_non_sys_data = pd.read_csv('pubmed_articles_test_non_sys.csv', engine='python', error_bad_lines=False)

In [37]:
len(test_non_sys_data)

623059

In [38]:
test_non_sys_data

,Unnamed: 0,title,abstract,authors,journal,journal_abbrv,year,mesh,month,volume,issue,pages,ptyp,pmid,registry_ids,dois
0,3,Mediating effects of resistance training skill...,"[{'header': '_UNSTRUCTURED', 'text': 'The purp...","[{'Initials': 'JJ', 'LastName': 'Smith', 'Fore...",Journal of sports sciences,J Sports Sci,2016,"['Adolescent', 'Body Composition', 'Body Fat D...",NaN,34,8,"{'page_from': '772', 'page_to': '79'}","['Journal Article', 'Randomized Controlled Tri...",26194449,[],['10.1080/02640414.2015.1069383']
1,7,Preventive Effects on Birth Outcomes: Bufferin...,"[{'header': 'OBJECTIVE', 'text': 'Although mat...","[{'Initials': 'ME', 'LastName': 'Feinberg', 'F...",Maternal and child health journal,Matern Child Health J,2016,"['Anxiety', 'Depression', 'Female', 'Humans', ...",Jan,20,1,"{'page_from': '56', 'page_to': '65'}","['Journal Article', 'Research Support, N.I.H.,...",26194453,['NCT01907412'],['10.1007/s10995-015-1801-3']
2,9,ESGAR consensus statement on liver MR imaging ...,"[{'header': 'OBJECTIVES', 'text': 'To develop ...","[{'Initials': 'E', 'LastName': 'Neri', 'ForeNa...",European radiology,Eur Radiol,2016,"['Adenoma, Liver Cell', 'Bile Ducts', 'Consens...",Apr,26,4,"{'page_from': '921', 'page_to': '9231'}","['Consensus Development Conference', 'Journal ...",26194455,[],['10.1007/s00330-015-3900-3']
3,10,Prevalence of degenerative and spondyloarthrit...,"[{'header': 'OBJECTIVE', 'text': 'To estimate ...","[{'Initials': 'B', 'LastName': 'Arnbak', 'Fore...",European radiology,Eur Radiol,2016,"['Adolescent', 'Adult', 'Age Factors', 'Female...",Apr,26,4,"{'page_from': '1191', 'page_to': '119203'}","['Journal Article', ""Research Support, Non-U.S...",26194456,[],['10.1007/s00330-015-3903-0']
4,21,Low bone mineral density and risk of incident ...,"[{'header': 'BACKGROUND', 'text': 'Prevalence ...","[{'Initials': 'L', 'LastName': 'Battalora', 'F...",Antiviral therapy,Antivir Ther,2016,"['Absorptiometry, Photon', 'Adult', 'Bone Dens...",NaN,21,1,"{'page_from': '45', 'page_to': '54'}","['Journal Article', ""Research Support, U.S. Go...",26194468,[],['10.3851/IMP2979']
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
623054,29995,Identification of donor splice sites using sup...,"[{'header': 'BACKGROUND', 'text': 'Identificat...","[{'Initials': 'PK', 'LastName': 'Meher', 'Fore...",Algorithms for molecular biology : AMB,Algorithms Mol Biol,2016,[],NaN,11,NaN,"{'page_from': '16', 'page_to': '16'}",['Journal Article'],27252772,[],['10.1186/s13015-016-0078-4']
623055,29996,Efficacy of extra-peritoneal pelvic packing in...,"[{'header': 'BACKGROUND', 'text': 'An option f...","[{'Initials': 'O', 'LastName': 'Chiara', 'Fore...",World journal of emergency surgery : WJES,World J Emerg Surg,2016,[],NaN,11,NaN,"{'page_from': '22', 'page_to': '22'}",['Journal Article'],27252773,[],['10.1186/s13017-016-0077-2']
623056,29997,Syrian refugees in Lebanon: the search for uni...,"[{'header': '_UNSTRUCTURED', 'text': 'The cris...","[{'Initials': 'K', 'LastName': 'Blanchet', 'Fo...",Conflict and health,Confl Health,2016,[],NaN,10,NaN,"{'page_from': '12', 'page_to': '12'}",['Journal Article'],27252775,[],['10.1186/s13031-016-0079-4']
623057,29998,Gastrodia and Uncaria (tianma gouteng) water e...,"[{'header': 'BACKGROUND', 'text': 'Gastrodia a...","[{'Initials': 'JW', 'LastName': 'Xian', 'ForeN...",Chinese medicine,Chin Med,2016,[],NaN,11,NaN,"{'page_from': '27', 'page_to': '27'}",['Journal Article'],27252774,[],['10.1186/s13020-016-0097-6']


In [39]:
test_sys_data = pd.read_csv('pubmed_articles_test_non_sys.csv', engine='python', error_bad_lines=False)

In [40]:
len(test_sys_data)

623059

In [41]:
test_sys_data

,Unnamed: 0,title,abstract,authors,journal,journal_abbrv,year,mesh,month,volume,issue,pages,ptyp,pmid,registry_ids,dois
0,3,Mediating effects of resistance training skill...,"[{'header': '_UNSTRUCTURED', 'text': 'The purp...","[{'Initials': 'JJ', 'LastName': 'Smith', 'Fore...",Journal of sports sciences,J Sports Sci,2016,"['Adolescent', 'Body Composition', 'Body Fat D...",NaN,34,8,"{'page_from': '772', 'page_to': '79'}","['Journal Article', 'Randomized Controlled Tri...",26194449,[],['10.1080/02640414.2015.1069383']
1,7,Preventive Effects on Birth Outcomes: Bufferin...,"[{'header': 'OBJECTIVE', 'text': 'Although mat...","[{'Initials': 'ME', 'LastName': 'Feinberg', 'F...",Maternal and child health journal,Matern Child Health J,2016,"['Anxiety', 'Depression', 'Female', 'Humans', ...",Jan,20,1,"{'page_from': '56', 'page_to': '65'}","['Journal Article', 'Research Support, N.I.H.,...",26194453,['NCT01907412'],['10.1007/s10995-015-1801-3']
2,9,ESGAR consensus statement on liver MR imaging ...,"[{'header': 'OBJECTIVES', 'text': 'To develop ...","[{'Initials': 'E', 'LastName': 'Neri', 'ForeNa...",European radiology,Eur Radiol,2016,"['Adenoma, Liver Cell', 'Bile Ducts', 'Consens...",Apr,26,4,"{'page_from': '921', 'page_to': '9231'}","['Consensus Development Conference', 'Journal ...",26194455,[],['10.1007/s00330-015-3900-3']
3,10,Prevalence of degenerative and spondyloarthrit...,"[{'header': 'OBJECTIVE', 'text': 'To estimate ...","[{'Initials': 'B', 'LastName': 'Arnbak', 'Fore...",European radiology,Eur Radiol,2016,"['Adolescent', 'Adult', 'Age Factors', 'Female...",Apr,26,4,"{'page_from': '1191', 'page_to': '119203'}","['Journal Article', ""Research Support, Non-U.S...",26194456,[],['10.1007/s00330-015-3903-0']
4,21,Low bone mineral density and risk of incident ...,"[{'header': 'BACKGROUND', 'text': 'Prevalence ...","[{'Initials': 'L', 'LastName': 'Battalora', 'F...",Antiviral therapy,Antivir Ther,2016,"['Absorptiometry, Photon', 'Adult', 'Bone Dens...",NaN,21,1,"{'page_from': '45', 'page_to': '54'}","['Journal Article', ""Research Support, U.S. Go...",26194468,[],['10.3851/IMP2979']
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
623054,29995,Identification of donor splice sites using sup...,"[{'header': 'BACKGROUND', 'text': 'Identificat...","[{'Initials': 'PK', 'LastName': 'Meher', 'Fore...",Algorithms for molecular biology : AMB,Algorithms Mol Biol,2016,[],NaN,11,NaN,"{'page_from': '16', 'page_to': '16'}",['Journal Article'],27252772,[],['10.1186/s13015-016-0078-4']
623055,29996,Efficacy of extra-peritoneal pelvic packing in...,"[{'header': 'BACKGROUND', 'text': 'An option f...","[{'Initials': 'O', 'LastName': 'Chiara', 'Fore...",World journal of emergency surgery : WJES,World J Emerg Surg,2016,[],NaN,11,NaN,"{'page_from': '22', 'page_to': '22'}",['Journal Article'],27252773,[],['10.1186/s13017-016-0077-2']
623056,29997,Syrian refugees in Lebanon: the search for uni...,"[{'header': '_UNSTRUCTURED', 'text': 'The cris...","[{'Initials': 'K', 'LastName': 'Blanchet', 'Fo...",Conflict and health,Confl Health,2016,[],NaN,10,NaN,"{'page_from': '12', 'page_to': '12'}",['Journal Article'],27252775,[],['10.1186/s13031-016-0079-4']
623057,29998,Gastrodia and Uncaria (tianma gouteng) water e...,"[{'header': 'BACKGROUND', 'text': 'Gastrodia a...","[{'Initials': 'JW', 'LastName': 'Xian', 'ForeN...",Chinese medicine,Chin Med,2016,[],NaN,11,NaN,"{'page_from': '27', 'page_to': '27'}",['Journal Article'],27252774,[],['10.1186/s13020-016-0097-6']


In [27]:
test_sys_data.sample(n=3, random_state=1)

NameError: name 'test_sys_data' is not defined